# CNN Food Classifier — Google Colab Training

**Model:** MobileNetV2 (pretrained on ImageNet) + custom classification head  
**Dataset:** Food-101 — 101 categories, 75,750 training images, 25,250 test images  
**Target:** ~70% validation accuracy after 10 epochs (transfer learning, frozen base)

### Instructions
1. `Runtime → Change runtime type → T4 GPU`
2. Run all cells top to bottom
3. Cell 4 mounts Google Drive — checkpoints are saved there so training survives session drops
4. If the session disconnects, re-run from Cell 4 — training resumes from the last checkpoint
5. Download `food_classifier.h5` from Google Drive when Cell 9 completes
6. Set a timer every **80 minutes** to keep the Colab session alive

In [ ]:
# Install dependencies and verify GPU
!pip install -q tensorflow tensorflow-datasets gradio pillow scikit-learn

import tensorflow as tf
print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

## Research Notes — Deployment Platform & Training Environment

When it came to deployment, I quickly realised that standard options weren't suitable for a TensorFlow model. Railway's free tier only provides 512MB RAM — insufficient given TensorFlow's ~300MB footprint alone. AWS Lambda and Azure Functions both have a 250MB package limit, which TensorFlow exceeds on its own. Through further research I found Hugging Face Spaces as an optimal solution: it is purpose-built for ML model deployment, offers a free tier with sufficient memory, has native Gradio support, and provides a public URL I could link directly from my portfolio. The trained model file (food_classifier.h5, ~14MB) is committed directly to the Space repository — no external storage required.

Before training I benchmarked three environments:
- Apple M1 16GB (CPU only): estimated 3–5 hours
- Apple M1 16GB (tensorflow-metal): estimated 45–90 minutes
- Google Colab free tier (T4 GPU): estimated 25–40 minutes

I selected Colab for speed. The main risk was session disconnection after ~90 minutes of inactivity, so I implemented checkpoint saving after every epoch. If the session dropped, training resumed from the last saved checkpoint. I set a timer every 80 minutes to keep the session alive during training.

In [ ]:
# Mount Google Drive for persistent checkpoint and model storage
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/food_classifier_checkpoints'
MODEL_SAVE_PATH = '/content/drive/MyDrive/food_classifier.h5'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoints will be saved to: {CHECKPOINT_DIR}')
print(f'Final model will be saved to: {MODEL_SAVE_PATH}')

In [ ]:
# Data loading and preprocessing
import tensorflow_datasets as tfds
import numpy as np

IMG_SIZE = 224
BATCH_SIZE = 16
NUM_CLASSES = 101

print('Loading Food-101 dataset (this may take a few minutes on first run)...')
(train_ds_raw, val_ds_raw), info = tfds.load(
    'food101',
    split=['train', 'validation'],
    as_supervised=True,
    with_info=True
)

class_names = info.features['label'].names
print(f'Classes: {NUM_CLASSES}  |  Train images: {info.splits["train"].num_examples}  |  Val images: {info.splits["validation"].num_examples}')

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

# Data augmentation layers applied inline during training
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

train_dataset = (
    train_ds_raw
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .map(lambda x, y: (augmentation(x, training=True), y),
         num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

val_dataset = (
    val_ds_raw
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print('Datasets ready.')

In [ ]:
# Model architecture — MobileNetV2 base + custom head
def build_model():
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )
    # Freeze all pretrained layers — we only train the classification head
    base_model.trainable = False

    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base_model(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)

    model = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_model()
model.summary()

In [ ]:
# Checkpoint loading — resumes training after a session drop
import glob

def load_latest_checkpoint():
    checkpoints = sorted(glob.glob(f'{CHECKPOINT_DIR}/epoch_*.h5'))
    if not checkpoints:
        return None, 0
    latest = checkpoints[-1]
    epoch_num = int(latest.split('epoch_')[1].replace('.h5', ''))
    print(f'Resuming from checkpoint: {latest} (epoch {epoch_num})')
    return tf.keras.models.load_model(latest), epoch_num

loaded_model, initial_epoch = load_latest_checkpoint()
if loaded_model is not None:
    model = loaded_model
    print(f'Resuming from epoch {initial_epoch + 1}')
else:
    initial_epoch = 0
    print('Starting training from scratch')

In [ ]:
# Training with checkpoint and early stopping callbacks
import time
EPOCHS = 10

checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath=f'{CHECKPOINT_DIR}/epoch_{{epoch:02d}}.h5',
    save_weights_only=False,
    save_best_only=False,
    verbose=1
)
early_stopping_cb = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=3,
    restore_best_weights=True
)

print(f'Training for up to {EPOCHS} epochs, starting from epoch {initial_epoch + 1}...')
t0 = time.time()

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    initial_epoch=initial_epoch,
    steps_per_epoch=500,
    validation_steps=100,
    callbacks=[checkpoint_cb, early_stopping_cb]
)

elapsed = (time.time() - t0) / 60
print(f'\nTraining complete in {elapsed:.1f} minutes')
print(f'Final train accuracy : {history.history["accuracy"][-1]:.4f}')
print(f'Final val accuracy   : {history.history["val_accuracy"][-1]:.4f}')

In [ ]:
# Save final model to Google Drive
model.save(MODEL_SAVE_PATH)
print(f'Model saved to {MODEL_SAVE_PATH}')
print('Download food_classifier.h5 from Google Drive to deploy to Hugging Face Spaces.')

In [ ]:
# Generate and display all 3 visualisations inline
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# --- Chart 1: Training curves ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['loss'], label='Train Loss', color='steelblue')
axes[0].plot(history.history['val_loss'], label='Val Loss', color='tomato')
axes[0].set_title('Loss per Epoch'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(history.history['accuracy'], label='Train Accuracy', color='steelblue')
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy', color='tomato')
axes[1].set_title('Accuracy per Epoch'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# --- Chart 2: Sample predictions ---
images_batch, labels_batch = next(iter(val_dataset.take(1)))
images_np = images_batch.numpy()[:9]
labels_np = labels_batch.numpy()[:9]
predictions = model.predict(images_np)
pred_indices = predictions.argmax(axis=1)

fig2, axes2 = plt.subplots(3, 3, figsize=(12, 12))
for i, ax in enumerate(axes2.flat):
    ax.imshow(images_np[i])
    true_label = class_names[labels_np[i]].replace('_', ' ').title()
    pred_label = class_names[pred_indices[i]].replace('_', ' ').title()
    conf = predictions[i][pred_indices[i]] * 100
    color = 'green' if pred_indices[i] == labels_np[i] else 'red'
    ax.set_title(f'True: {true_label}\nPred: {pred_label} ({conf:.1f}%)', color=color, fontsize=9)
    ax.axis('off')
plt.suptitle('Sample Predictions', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

# --- Chart 3: Confusion matrix (top 20 most confused) ---
all_true, all_pred = [], []
for imgs, lbls in val_dataset:
    preds = model.predict(imgs, verbose=0)
    all_pred.extend(preds.argmax(axis=1))
    all_true.extend(lbls.numpy())
cm = confusion_matrix(np.array(all_true), np.array(all_pred))
off_diag = cm.copy(); np.fill_diagonal(off_diag, 0)
top20 = off_diag.sum(axis=1).argsort()[-20:][::-1]
cm_sub = cm[np.ix_(top20, top20)]
labels_sub = [class_names[i].replace('_', ' ').title() for i in top20]
plt.figure(figsize=(16, 13))
sns.heatmap(cm_sub, annot=True, fmt='d', cmap='Blues', xticklabels=labels_sub, yticklabels=labels_sub)
plt.title('Confusion Matrix — Top 20 Most Confused Classes')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.xticks(rotation=45, ha='right', fontsize=8); plt.yticks(fontsize=8)
plt.tight_layout(); plt.show()

## Download & Deploy

### Download the trained model
1. Open Google Drive at [drive.google.com](https://drive.google.com)
2. Find `food_classifier.h5` in the root of My Drive
3. Right-click → Download

### Deploy to Hugging Face Spaces
```bash
# 1. Install Hugging Face CLI
pip install huggingface_hub

# 2. Log in
huggingface-cli login

# 3. Clone the Space repository
git clone https://huggingface.co/spaces/xavier-oc-machinelearn/cnn-food-classifier hf-space

# 4. Copy project files into the Space
cp train.py app.py notebook.ipynb README.md requirements.txt food_classifier.h5 hf-space/
cp -r plots/ hf-space/plots/

# 5. Push to Hugging Face
cd hf-space
git add .
git commit -m "Deploy CNN food classifier"
git push
```

The app will be live at: https://huggingface.co/spaces/xavier-oc-machinelearn/cnn-food-classifier

### Resume training after a session drop
Simply re-run this notebook from Cell 4. The checkpoint loader will detect the latest
saved epoch from Google Drive and resume training from there automatically.